# PAP Emulation - FMI / FMPy

Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary. Licensed under GNU GPL v3.0. Third-party dependencies retain their own licenses.

FMI 1.0/2.0/3.0 model-description inspection and FMU execution using FMPy. Exports normalized time-series values while requiring a separate typed SIR port mapping.

**Author:** Abhishek Choudhary · **Affiliation:** AyeAI · **ORCID:** [0009-0002-0684-8320](https://orcid.org/0009-0002-0684-8320)

**Deployment:** browser-only platform; this notebook runs in a user-controlled Colab runtime. GPU/TPU availability is variable and not guaranteed. **Evidence boundary:** results remain `SIMULATED` and `UNVERIFIED`; this is not HIL or physical verification.

## 0. Runtime, security and provenance
Run cells in order. The notebook records backend versions, solver settings, runtime device, model configuration, timestamps and SHA-256 job/result digests. Results must be imported through the separate PAP Colab Bridge. Never upload untrusted native FMUs or binaries.

In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.
# AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
# Embedded in each notebook so notebooks remain independently runnable.
PAP_BRIDGE_VERSION = '1.0.0'
PAP_JOB_SCHEMA = 'pap.emulation.job/1.0'
PAP_RESULT_SCHEMA = 'pap.emulation.result/1.0'

def pap_canonical(value):
    import json
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False)

def pap_digest(text):
    import hashlib
    return 'sha256:' + hashlib.sha256(text.encode('utf-8')).hexdigest()

def pap_make_job(backend_id, operation, inputs):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_JOB_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'createdAt': datetime.now(timezone.utc).isoformat(),
        'target': {'backendId': backend_id, 'backendVersion': None, 'operation': operation, 'requestedDevice': 'auto'},
        'sir': {'header': {'id': 'colab-reference-fixture', 'irVersion': '1.0.0'}, 'components': []},
        'inputs': inputs,
        'observations': None,
        'configuration': {},
        'provenance': {'source': 'COLAB_REFERENCE_FIXTURE', 'sourceVersion': '1.0.0', 'license': 'GPL-3.0-only'}
    }
    canonical = pap_canonical(body)
    digest = pap_digest(canonical)
    return {**body, 'jobId': 'papjob-' + digest[7:27], 'digest': digest, 'canonicalPayload': canonical}

def pap_verify_job(job, expected_backend):
    import json
    if not isinstance(job, dict) or job.get('schema') != PAP_JOB_SCHEMA:
        raise ValueError('Unsupported/missing PAP job schema')
    payload = job.get('canonicalPayload')
    if not isinstance(payload, str): raise ValueError('Job canonicalPayload missing')
    parsed = json.loads(payload)
    body = {k:v for k,v in job.items() if k not in ('jobId','digest','canonicalPayload')}
    if pap_canonical(parsed) != pap_canonical(body): raise ValueError('Job payload/body mismatch')
    digest = pap_digest(payload)
    if digest != job.get('digest') or job.get('jobId') != 'papjob-' + digest[7:27]:
        raise ValueError('Job digest/jobId verification failed')
    if job.get('target',{}).get('backendId') != expected_backend:
        raise ValueError('Job targets a different backend')
    if not isinstance(job.get('inputs'), dict): raise ValueError('Job inputs must be an object')
    return job

def pap_get_job(backend_id, operation, default_inputs, use_demo=True):
    if use_demo:
        return pap_make_job(backend_id, operation, default_inputs), True
    from google.colab import files
    import json
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith('.json')]
    if len(candidates) != 1: raise ValueError('Upload exactly one browser-exported PAP job .json')
    return pap_verify_job(json.loads(uploaded[candidates[0]].decode('utf-8')), backend_id), False

def pap_make_result(job, backend_id, backend_version, outputs, diagnostics, device='cpu'):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_RESULT_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'jobId': job['jobId'],
        'jobDigest': job['digest'],
        'backend': {'id': backend_id, 'version': str(backend_version)},
        'execution': {'status': 'EXECUTED', 'runtime': 'COLAB', 'startedAt': None,
                      'finishedAt': datetime.now(timezone.utc).isoformat(), 'device': str(device)},
        'outputs': outputs,
        'diagnostics': diagnostics,
        'fidelity': 'SIMULATED',
        'physicalVerification': False,
        'evidenceState': 'UNVERIFIED',
        'claims': {'verified': False, 'hil': False, 'physical': False}
    }
    canonical = pap_canonical(body)
    return {**body, 'canonicalPayload': canonical, 'resultDigest': pap_digest(canonical)}

def pap_export(job, result, is_demo, result_filename='pap-emulation-result.json'):
    import json
    with open(result_filename, 'w', encoding='utf-8') as stream:
        json.dump(result, stream, indent=2, ensure_ascii=False, allow_nan=False)
    if is_demo:
        with open('pap-emulation-job.json', 'w', encoding='utf-8') as stream:
            json.dump(job, stream, indent=2, ensure_ascii=False, allow_nan=False)
    try:
        from google.colab import files
        files.download(result_filename)
        if is_demo: files.download('pap-emulation-job.json')
    except ImportError:
        print('Saved locally:', result_filename, '(download manually outside Colab)')
    print('Result remains SIMULATED / UNVERIFIED. Import it through the PAP Colab Bridge.')


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
%pip -q install 'fmpy[complete]' 'numpy>=1.26,<3'

In [ ]:
import os, json, hashlib, platform, sys
import numpy as np, fmpy
from fmpy import read_model_description, simulate_fmu
print({'python':sys.version,'platform':platform.platform(),'fmpy':getattr(fmpy,'__version__','unknown')})
PAP_USE_DEMO_JOB=True
default_inputs={'startTime':0.0,'stopTime':1.0,'outputInterval':0.01,'outputVariables':[]}
job,job_is_demo=pap_get_job('fmpy.fmi','fmpy.simulate',default_inputs,PAP_USE_DEMO_JOB)

## 1. Upload one trusted FMU
FMUs can contain native executable code. Only use trusted FMUs in a disposable runtime. Colab is not asserted to be a hardened sandbox for hostile native binaries.

In [ ]:
from google.colab import files
uploaded=files.upload(); fmus=[name for name in uploaded if name.lower().endswith('.fmu')]
if len(fmus)!=1: raise ValueError('Upload exactly one trusted .fmu file')
fmu_path=fmus[0]; fmu_sha=hashlib.sha256(open(fmu_path,'rb').read()).hexdigest(); md=read_model_description(fmu_path)
print({'sha256':fmu_sha,'fmiVersion':md.fmiVersion,'modelName':md.modelName,'guid':md.guid,'hasCoSimulation':bool(md.coSimulation),'hasModelExchange':bool(md.modelExchange)})

## 2. Inspect, simulate and export
Review causality, units, variability and value references. The raw result cannot be treated as canonical until the adapter maps each FMU variable to a SIR port, unit and coordinate frame.

In [ ]:
variables=[v.name for v in md.modelVariables if v.causality=='output']
if not variables: raise ValueError('FMU has no output variables')
selected=job['inputs'].get('outputVariables') or variables[:min(8,len(variables))]
if set(selected)-set(variables): raise ValueError('Requested FMU output name is not declared')
start=float(job['inputs'].get('startTime',0)); stop=float(job['inputs'].get('stopTime',1)); interval=float(job['inputs'].get('outputInterval',0.01))
if not stop>start or interval<=0: raise ValueError('Invalid start/stop/output interval')
data=simulate_fmu(fmu_path,start_time=start,stop_time=stop,output=selected,output_interval=interval)
if len(data)==0: raise RuntimeError('FMU returned no samples')
outputs=[]
for i,row in enumerate(data):
    values={name:float(row[name]) for name in selected}
    if not np.isfinite(list(values.values())).all(): raise RuntimeError('FMU produced non-finite output')
    outputs.append({'sequence':int(i),'time':float(row['time']),'values':values})
result=pap_make_result(job,'fmpy.fmi',getattr(fmpy,'__version__','unknown'),outputs,{'fmiVersion':md.fmiVersion,'modelName':md.modelName,'guid':md.guid,'fmuSha256':fmu_sha,'selectedVariables':selected,'mappingRequired':True},'cpu')
pap_export(job,result,job_is_demo,'pap-fmpy-result.json')